# Figure 6 — LinkD-Agent interactive analysis & benchmark

## Data availability

All inputs for this notebook are **copied into** `For Reviewer/source_data/` (or shown from `illustrations/` when a panel cannot be recomputed).

- No Zenodo download is required.
- No paths outside `For Reviewer/` are used after packaging.
- See `DATA_AVAILABILITY.md` and `source_data/manifest.csv` for origins and checksums.

**Files used below** are listed in each panel section.

- `source_data/benchmark/` summary JSONL + leaderboard

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from linkd_repro import paths, style, io, illustrate
style.apply()
paths.ensure_output_dirs()
print("For Reviewer root:", paths.ROOT)
print("source_data OK:", paths.SOURCE.exists())

## Panels a–b — Architecture / planning schematics

In [ ]:
illustrate.show_panel('fig6_a', title='Panel a')
illustrate.show_panel('fig6_b', title='Panel b')

## Panel c — Agent benchmark heatmap

In [ ]:

import json
# Aggregate summary.*.jsonl into a method x task score matrix when possible
rows = []
for p in sorted((paths.SOURCE / "benchmark").glob("summary.*.jsonl")):
    with p.open() as f:
        for line in f:
            line=line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except Exception:
                continue
            rows.append(rec)
sumdf = pd.DataFrame(rows)
display(Markdown(f"Loaded {len(sumdf)} summary records from benchmark JSONL"))
# Heuristic column detection
score_col = next((c for c in ["primary_score", "score", "metric_value", "ndcg", "auroc", "cindex"] if c in sumdf.columns), None)
method_col = next((c for c in ["condition", "method", "agent", "model"] if c in sumdf.columns), None)
task_col = next((c for c in ["scenario", "task", "task_id"] if c in sumdf.columns), None)
fig, ax = plt.subplots(figsize=(6.5, 3.8))
if score_col and method_col and task_col and len(sumdf):
    piv = sumdf.pivot_table(index=method_col, columns=task_col, values=score_col, aggfunc="mean")
    im = ax.imshow(piv.values, aspect="auto", cmap="YlOrRd")
    ax.set_xticks(range(len(piv.columns)))
    ax.set_xticklabels(piv.columns, rotation=45, ha="right", fontsize=6)
    ax.set_yticks(range(len(piv.index)))
    ax.set_yticklabels(piv.index, fontsize=6)
    fig.colorbar(im, ax=ax, fraction=0.046)
    src = piv.reset_index()
else:
    # Fallback: show leaderboard.csv
    lb = paths.SOURCE / "benchmark" / "leaderboard.csv"
    if lb.exists():
        ldb = pd.read_csv(lb)
        display(ldb.head())
        ax.axis("off")
        ax.table(cellText=ldb.head(12).values, colLabels=list(ldb.columns), loc="center", fontsize=6)
        src = ldb
    else:
        ax.text(0.5, 0.5, "No benchmark matrix columns found", ha="center")
        src = sumdf.head(0)
ax.set_title("Fig 6c — benchmark summary")
fig.tight_layout()
out = style.save_panel(fig, "fig6_c_benchmark", src if isinstance(src, pd.DataFrame) else pd.DataFrame())
plt.show()
print(out)
# Also note pre-rendered reference figures were copied if present
for name in ["fig6_benchmark_heatmap.png", "fig6_benchmark_bars.png"]:
    p = paths.SOURCE / "benchmark" / name
    if p.exists():
        display(Markdown(f"Reference render available: `{p.relative_to(paths.ROOT)}`"))
